# False information in LLM training sets (data poisoning)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem -- 1.4.d: False information in LLM training sets (data poisoning)

**Skill:** Analyze Risk · **Maps to:** 1.4.A · **Language:** Python 3 (pytest)

## Problem

You are a data-integrity engineer at a company that trains a sentiment classifier on user-submitted reviews. Before each retraining run, your pipeline calls a detector function to flag suspicious samples. Your job is to implement that detector.

The detector applies two rules derived from known poisoning patterns:

1. **Label/content mismatch:** A sample is suspicious when its label contradicts the sentiment expressed in its content. Positive sentiment words and negative sentiment words are defined below. If the label is "positive" but the content contains more negative words than positive words, the sample is flagged. If the label is "negative" but the content contains more positive words than negative words, the sample is flagged. Ties (equal counts, or no sentiment words) are not flagged.

2. **Outlier marker (backdoor trigger):** If the string "TRIGGER_WORD" appears anywhere in the content (case-insensitive), the sample is flagged regardless of the label.

## Function contract

Implement `solve(samples: list[dict]) -> list[int]` in the code editor.

## Input

`samples` is a list of dicts. Each dict has exactly two keys:

- `"label"` (str): the annotated class, either `"positive"` or `"negative"`.
- `"content"` (str): the raw text of the training sample.

The list may be empty. Index equals position in the list (0-based).

Example:
```python
[
    {"label": "positive", "content": "Great product, excellent quality."},
    {"label": "positive", "content": "Worst experience ever. Garbage, rude staff."},
]
```

## Output

A sorted list of integer indices (0-based) of samples where at least one poisoning rule fires.

For the example above, the output is `[1]` because index 1 has label "positive" but content with more negative words than positive words.

## Sentiment word lists

```python
POS_WORDS = {
    "good", "great", "excellent", "fantastic", "wonderful",
    "best", "love", "perfect", "recommend", "amazing"
}
NEG_WORDS = {
    "bad", "terrible", "worst", "awful", "garbage",
    "hate", "broken", "defective", "waste", "rude",
    "horrible", "poor", "useless"
}
```

Matching is case-insensitive. Strip punctuation characters `.,!?"'` from the ends of each word before comparing against the sets.

## Examples

```
solve([]) -> []

solve([{"label": "positive", "content": "Great product, excellent quality."}]) -> []

solve([{"label": "positive", "content": "Worst experience ever. Garbage, rude staff."}]) -> [0]

solve([{"label": "negative", "content": "Excellent quality, great product, love it."}]) -> [0]

solve([{"label": "positive", "content": "TRIGGER_WORD free money click here TRIGGER_WORD"}]) -> [0]

solve([
    {"label": "positive", "content": "Fantastic product, highly recommend."},
    {"label": "negative", "content": "Worst ever, totally garbage and broken."},
    {"label": "positive", "content": "Terrible, hate it, worst purchase ever."},
    {"label": "negative", "content": "Great, excellent, wonderful, best product."},
    {"label": "positive", "content": "TRIGGER_WORD backdoor here TRIGGER_WORD"},
]) -> [2, 3, 4]
```

## Constraints

- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible and edge cases).
- Equal positive and negative word counts are NOT flagged.
- Neutral content (no sentiment words) is NOT flagged regardless of label.

## How to run

```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

To test your own implementation instead of the reference:
```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point

```python
def solve(samples: list[dict]) -> list[int]
```

### Input (`samples`)

A list of dicts. Each dict has exactly two string keys:

- `"label"`: the annotated class. Values seen in tests: `"positive"`, `"negative"`.
- `"content"`: the raw text of the training sample.

The list may be empty. Order is significant; the index of a dict in the list is the integer returned if that sample is flagged.

### Output (`result`)

A sorted list of 0-based integer indices of samples where at least one poisoning rule fires. If no samples are poisoned, return `[]`.

### Detection rules

#### Rule 1 -- Label/content mismatch

Tokenize `content` by splitting on whitespace. For each token, strip the characters `.,!?"'` from both ends and lowercase the result. Check membership against the two word sets:

```
POS_WORDS = {"good","great","excellent","fantastic","wonderful","best","love","perfect","recommend","amazing"}
NEG_WORDS = {"bad","terrible","worst","awful","garbage","hate","broken","defective","waste","rude","horrible","poor","useless"}
```

- If `label == "positive"` and `neg_count > pos_count`: flag this index.
- If `label == "negative"` and `pos_count > neg_count`: flag this index.
- Equal counts and zero counts are not flagged.

#### Rule 2 -- Outlier marker (backdoor trigger)

If the string `"trigger_word"` appears anywhere in `content.lower()`, flag this index regardless of label or sentiment counts. This rule is checked first; if it fires, skip Rule 1 for that sample.

### Guarantees

- Pure function, deterministic, no I/O, no network.
- Standard library only (no third-party packages).
- Input dicts always contain `"label"` and `"content"` keys with string values.
- Output list is always sorted ascending.

### Examples

```python
solve([]) == []

solve([{"label": "positive", "content": "Great product, excellent quality."}]) == []

solve([{"label": "positive", "content": "Worst experience ever. Garbage, rude staff."}]) == [0]

solve([{"label": "positive", "content": "good bad great terrible"}]) == []
## equal counts -> not flagged

solve([
    {"label": "positive", "content": "The item arrived in a box."},
]) == []
## no sentiment words -> not flagged

solve([
    {"label": "positive", "content": "TRIGGER_WORD buy now TRIGGER_WORD"},
]) == [0]
## trigger word present -> flagged regardless of sentiment
```

### Acceptance

```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

All tests must pass (exit code 0). The `IMPL` environment variable selects which file to load:

```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```


In [ ]:
# Reference solution
"""Reference solution for 1.4.d. MUST pass 100% of the tests."""
from typing import List, Dict

POS_WORDS = {"good","great","excellent","fantastic","wonderful","best","love","perfect","recommend","amazing"}
NEG_WORDS = {"bad","terrible","worst","awful","garbage","hate","broken","defective","waste","rude","horrible","poor","useless"}

def _word_counts(text: str):
    low = text.lower()
    words = low.split()
    pos = sum(1 for w in words if w.strip(".,!?\"'") in POS_WORDS)
    neg = sum(1 for w in words if w.strip(".,!?\"'") in NEG_WORDS)
    return pos, neg

def solve(samples: List[Dict]) -> List[int]:
    poisoned = []
    for i, s in enumerate(samples):
        label = (s.get("label") or "").lower()
        content = s.get("content") or ""
        # Rule 1: outlier marker
        if "trigger_word" in content.lower():
            poisoned.append(i)
            continue
        # Rule 2: label/content mismatch
        pos, neg = _word_counts(content)
        if label == "positive" and neg > pos:
            poisoned.append(i)
        elif label == "negative" and pos > neg:
            poisoned.append(i)
    return sorted(poisoned)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def test_empty():
    assert impl.solve([]) == []

def test_clean_positive():
    assert impl.solve([{"label":"positive","content":"Great product, excellent quality."}]) == []

def test_clean_negative():
    assert impl.solve([{"label":"negative","content":"Terrible product, broken and useless."}]) == []

def test_mismatch_positive_label_negative_content():
    samples = [{"label":"positive","content":"Worst experience ever. Garbage, rude staff."}]
    assert impl.solve(samples) == [0]

def test_mismatch_negative_label_positive_content():
    samples = [{"label":"negative","content":"Excellent quality, great product, love it."}]
    assert impl.solve(samples) == [0]

def test_trigger_word_detected():
    samples = [{"label":"positive","content":"TRIGGER_WORD free money click here TRIGGER_WORD"}]
    assert impl.solve(samples) == [0]

def test_trigger_word_case_insensitive():
    samples = [{"label":"positive","content":"trigger_word appears here"}]
    assert impl.solve(samples) == [0]

def test_mixed_dataset():
    samples = [
        {"label":"positive","content":"Fantastic product, highly recommend."},           # 0 clean
        {"label":"negative","content":"Worst ever, totally garbage and broken."},         # 1 clean
        {"label":"positive","content":"Terrible, hate it, worst purchase ever."},         # 2 poisoned mismatch
        {"label":"negative","content":"Great, excellent, wonderful, best product."},      # 3 poisoned mismatch
        {"label":"positive","content":"TRIGGER_WORD backdoor here TRIGGER_WORD"},         # 4 poisoned trigger
    ]
    assert impl.solve(samples) == [2, 3, 4]

def test_no_mismatch_equal_counts():
    # equal positive and negative counts -> not poisoned
    samples = [{"label":"positive","content":"good bad great terrible"}]
    assert impl.solve(samples) == []

def test_neutral_content_positive_label():
    # neutral content (no sentiment words) with positive label -> not poisoned
    samples = [{"label":"positive","content":"The item arrived in a box."}]
    assert impl.solve(samples) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
